# Training a Tiny RNN Language Model

This notebook shows the same process from the slide:

1. Start with a corpus of words: `x(1), x(2), x(3), ..., x(T)`.
2. Feed the words into an RNN language model.
3. At each time step `t`, predict a probability distribution over the next word.
4. Compute cross-entropy loss for each step:

   `J(t) = -log probability assigned to the true next word`

5. Average those losses to get the overall training loss:

   `J(theta) = (1 / T) * sum_t J(t)`

## 1. Import PyTorch

We use PyTorch because it already provides embeddings, RNNs, cross-entropy loss, and optimizers.

In [1]:
from __future__ import annotations

import torch
from torch import nn

torch.manual_seed(7)

## 2. Build a Tiny Corpus

The corpus is just a sequence of words. For language modeling, each word is used to predict the next word.

In [2]:
corpus = "the students opened their exams and the students studied their notes"
words = corpus.lower().split()

print(words)

['the', 'students', 'opened', 'their', 'exams', 'and', 'the', 'students', 'studied', 'their', 'notes']


## 3. Create a Vocabulary and Encode Words

Neural networks do not directly read strings, so each word is mapped to an integer ID.

In [3]:
vocab = sorted(set(words))
word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for word, i in word_to_id.items()}

encoded = torch.tensor([word_to_id[word] for word in words], dtype=torch.long)

print("Vocabulary:")
print(word_to_id)
print()
print("Encoded corpus:")
print(encoded)

Vocabulary:
{'and': 0, 'exams': 1, 'notes': 2, 'opened': 3, 'students': 4, 'studied': 5, 'the': 6, 'their': 7}

Encoded corpus:
tensor([6, 4, 3, 7, 1, 0, 6, 4, 5, 7, 2])


## 4. Create Input and Target Sequences

At time step `t`, the input is the current word `x(t)`.

The target is the true next word `x(t+1)`.

In [4]:
input_ids = encoded[:-1]
target_ids = encoded[1:]

print("Training pairs:")
for current_id, next_id in zip(input_ids, target_ids):
    print(f"{id_to_word[current_id.item()]:8s} -> {id_to_word[next_id.item()]}")

Training pairs:
the      -> students
students -> opened
opened   -> their
their    -> exams
exams    -> and
and      -> the
the      -> students
students -> studied
studied  -> their
their    -> notes


## 5. Define the RNN Language Model

The model has three parts:

- `Embedding`: converts each word ID into a dense vector.
- `RNN`: reads the word vectors one time step at a time.
- `Linear`: converts each hidden state into vocabulary scores.

Those vocabulary scores are called `logits`. After `softmax`, they become the predicted probability distribution `y_hat(t)`.

In [5]:
class TinyRNNLanguageModel(nn.Module):
    """Embedding -> RNN -> vocabulary scores for every time step."""

    def __init__(self, vocab_size: int, embedding_dim: int, hidden_dim: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
        )
        self.output_layer = nn.Linear(hidden_dim, vocab_size)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        # token_ids shape: [batch_size, sequence_length]
        embeddings = self.embedding(token_ids)
        hidden_states, _ = self.rnn(embeddings)
        logits = self.output_layer(hidden_states)

        # logits shape: [batch_size, sequence_length, vocab_size]
        return logits


model = TinyRNNLanguageModel(
    vocab_size=len(vocab),
    embedding_dim=8,
    hidden_dim=16,
)

print(model)

TinyRNNLanguageModel(
  (embedding): Embedding(8, 8)
  (rnn): RNN(8, 16, batch_first=True)
  (output_layer): Linear(in_features=16, out_features=8, bias=True)
)


## 6. Add the Batch Dimension

PyTorch RNNs usually process a batch of sequences.

With `batch_first=True`, the expected shape is:

`[batch_size, sequence_length]`

Here we have one sequence, so `batch_size = 1`.

In [6]:
x = input_ids.unsqueeze(0)
y = target_ids.unsqueeze(0)

print("x shape:", x.shape)
print("y shape:", y.shape)

x shape: torch.Size([1, 10])
y shape: torch.Size([1, 10])


## 7. Train the Model

At each epoch:

1. Run the whole sequence through the RNN.
2. Predict vocabulary scores at every time step.
3. Compare those scores with the true next-word IDs.
4. Backpropagate the average cross-entropy loss.

Important shape detail:

- The model returns logits as `[batch, time, vocab_size]`.
- `CrossEntropyLoss` expects `[batch, vocab_size, time]`.
- So we use `logits.transpose(1, 2)`.

In [7]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_function = nn.CrossEntropyLoss()

loss_history = []

for epoch in range(1, 151):
    logits = model(x)

    # CrossEntropyLoss computes the average over all time steps:
    # J(theta) = (1 / T) * sum_t J(t)
    loss = loss_function(logits.transpose(1, 2), y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if epoch == 1 or epoch % 30 == 0:
        print(f"epoch {epoch:3d} | average sequence loss J(theta): {loss.item():.4f}")

epoch   1 | average sequence loss J(theta): 1.8570
epoch  30 | average sequence loss J(theta): 0.0013
epoch  60 | average sequence loss J(theta): 0.0004
epoch  90 | average sequence loss J(theta): 0.0003
epoch 120 | average sequence loss J(theta): 0.0003
epoch 150 | average sequence loss J(theta): 0.0002


## 8. Inspect Step-by-Step Predictions

For each time step, we can look at:

- the current input word `x(t)`
- the true next word `x(t+1)`
- the model's predicted next word
- the probability assigned to the true next word
- the step loss `J(t)`

In [8]:
model.eval()

with torch.no_grad():
    logits = model(x)
    probabilities = torch.softmax(logits, dim=-1)
    predicted_ids = probabilities.argmax(dim=-1).squeeze(0)

print("Next-word predictions after training:")
for t, (current_id, true_next_id, predicted_id) in enumerate(
    zip(input_ids, target_ids, predicted_ids),
    start=1,
):
    current_word = id_to_word[current_id.item()]
    true_next_word = id_to_word[true_next_id.item()]
    predicted_word = id_to_word[predicted_id.item()]
    probability_of_true_next_word = probabilities[0, t - 1, true_next_id].item()
    step_loss = -torch.log(probabilities[0, t - 1, true_next_id]).item()

    print(
        f"t={t:2d} | x(t)='{current_word:8s}' "
        f"| true x(t+1)='{true_next_word:8s}' "
        f"| predicted='{predicted_word:8s}' "
        f"| p(true next word)={probability_of_true_next_word:.3f} "
        f"| J(t)={step_loss:.3f}"
    )

Next-word predictions after training:
t= 1 | x(t)='the     ' | true x(t+1)='students' | predicted='students' | p(true next word)=1.000 | J(t)=0.000
t= 2 | x(t)='students' | true x(t+1)='opened  ' | predicted='opened  ' | p(true next word)=1.000 | J(t)=0.000
t= 3 | x(t)='opened  ' | true x(t+1)='their   ' | predicted='their   ' | p(true next word)=1.000 | J(t)=0.000
t= 4 | x(t)='their   ' | true x(t+1)='exams   ' | predicted='exams   ' | p(true next word)=1.000 | J(t)=0.000
t= 5 | x(t)='exams   ' | true x(t+1)='and     ' | predicted='and     ' | p(true next word)=1.000 | J(t)=0.000
t= 6 | x(t)='and     ' | true x(t+1)='the     ' | predicted='the     ' | p(true next word)=1.000 | J(t)=0.000
t= 7 | x(t)='the     ' | true x(t+1)='students' | predicted='students' | p(true next word)=1.000 | J(t)=0.000
t= 8 | x(t)='students' | true x(t+1)='studied ' | predicted='studied ' | p(true next word)=1.000 | J(t)=0.000
t= 9 | x(t)='studied ' | true x(t+1)='their   ' | predicted='their   ' | p(true ne

## 9. Generate Text

After training, we can start with a short prompt and repeatedly feed the model's predicted word back into the sequence.

In [9]:
prompt = "the students"
generated_words = prompt.split()

with torch.no_grad():
    for _ in range(6):
        prompt_ids = torch.tensor(
            [[word_to_id[word] for word in generated_words]],
            dtype=torch.long,
        )
        next_word_logits = model(prompt_ids)[0, -1]
        next_word_id = torch.argmax(next_word_logits).item()
        generated_words.append(id_to_word[next_word_id])

print("Generated text:")
print(" ".join(generated_words))

Generated text:
the students opened their exams and the students
